<div style="text-align: center;"><img src="https://www.databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png" alt="Databricks Learning" width="600"></div>


# Lecture — Fundamentals of Data Preparation

## Overview

This lecture introduces the workflow you follow in every demo and lab in this module, **as you do it on Databricks**: clean the data with Spark, split it, and put every step that learns from the data into a Spark ML **Pipeline** that is fit on the training set only.

This lecture covers the following four sections that build on each other:

- **A. The Data Preparation Workflow**. Where data prep sits in the ML lifecycle, the difference between cleaning and feature engineering, and how to detect and treat outliers
- **B. Feature Engineering and Feature Selection**. Transforming raw data into informative features, choosing which features to keep, and building leakage-safe Spark ML pipelines
- **C. Splitting Data**. Train/validation/test splits, cross-validation, time-based and group-based splits, and avoiding data leakage
- **D. Sampling Methods**. Random, stratified, and cluster sampling with Spark DataFrame APIs

## Learning Objectives

By the end of this lecture, you will be able to:

- Describe where data preparation sits in the iterative ML lifecycle, and the correct order of cleaning, splitting, and fitting
- Distinguish cleaning and formatting from feature engineering and feature selection
- Describe how to detect outliers with domain rules, IQR, or z-scores, and choose an appropriate treatment
- Choose an appropriate split strategy, including random, time-based, and group-based splits, and explain cross-validation
- Explain why every data-dependent step belongs in a Spark ML Pipeline that is fit on the training set only
- Describe how to draw random, stratified, and cluster samples with Spark (`sample`, `sampleBy`, and sampled group IDs)

## A. The Data Preparation Workflow

### A1. Where Data Prep Sits in the Machine Learning Lifecycle

Machine learning is an **iterative** lifecycle: models are developed, deployed, monitored, and then improved, and what you learn in production often sends you back to data preparation.
* This module covers the "Data Preprocessing / Feature Engineering". 
* Module 1 covered data collection (loading data into Delta + UC). 
* Model training, evaluation, deployment, and monitoring are covered in follow-on courses, such as *Machine Learning Model Development* and *Machine Learning Model Deployment*.

Data preparation can be a substantial part of an ML project, and its quality directly affects model quality.

<div style="max-width: 1000px; margin: 0 auto;">
<svg width="100%" viewBox="0 0 800 525" role="img" style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;">
  <title>The Machine Learning Lifecycle — Data Preparation Context</title>
  <desc>Two phases of the Machine Learning lifecycle. Model Development contains Business problem, Data collection, and Data prep / feature engineering in a top row, with Model training and Model evaluation in a bottom row connected by a downward then leftward arrow. An arrow exits Model evaluation rightward into the Production phase, which contains Model deployment, Model monitoring, and Model retrain in sequence.</desc>
  <defs>
    <marker id="mlc-arrow" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse">
      <path d="M2 1L8 5L2 9" fill="none" stroke="#1B3139" stroke-width="1.5" stroke-linecap="round" stroke-linejoin="round"/>
    </marker>
  </defs>
  <!-- Top subgroup: Model Development — fill then dashed outline -->
  <rect x="30" y="52" width="700" height="250" rx="12" fill="#2272B4" fill-opacity="0.08"/>
  <rect x="30" y="52" width="700" height="250" rx="12" fill="none" stroke="#2272B4" stroke-width="1.5" stroke-dasharray="6 3"/>
  <text x="50" y="200" font-size="13" font-weight="bold" fill="#0b2026">Model Development</text>
  <text x="50" y="215" font-size="11" fill="#618794">(static historical data)</text>
  <!-- 1st step: Business problem -->
  <rect x="43" y="98" width="115" height="52" rx="8" fill="#ffffff" stroke="#90A5B1" stroke-width="1.2"/>
  <text x="100" y="120" text-anchor="middle" font-size="11.5" font-weight="600" fill="#0b2026">Business</text>
  <text x="100" y="138" text-anchor="middle" font-size="11.5" font-weight="600" fill="#0b2026">Problem</text>
  <!-- 2nd step:  Define success criteria -->
  <rect x="175" y="98" width="115" height="52" rx="8" fill="#ffffff" stroke="#90A5B1" stroke-width="1.2"/>
  <text x="232" y="120" text-anchor="middle" font-size="11.5" font-weight="600" fill="#0b2026">Define Success</text>
  <text x="232" y="138" text-anchor="middle" font-size="11.5" font-weight="600" fill="#0b2026">Criteria</text>
  <!-- 3rd step: Data collection -->
  <rect x="305" y="98" width="115" height="52" rx="8" fill="#ffffff" stroke="#90A5B1" stroke-width="1.5"/>
  <text x="360" y="125" text-anchor="middle" font-size="11.5" font-weight="600" fill="#0b2026">Data Collection</text>
  <!-- 4th step: Data prepprocessing / feature engineering (highlighted) -->
  <rect x="435" y="98" width="150" height="52" rx="8" fill="#00A972" fill-opacity="0.14" stroke="#00A972" stroke-width="2"/>
  <text x="510" y="120" text-anchor="middle" font-size="11.5" font-weight="600" fill="#0b2026">Data Preprocessing /</text>
  <text x="510" y="136" text-anchor="middle" font-size="11.5" font-weight="600" fill="#0b2026">Feature Engineering</text>
  <!-- 5th step: Model training -->
  <rect x="605" y="98" width="115" height="52" rx="8" fill="#ffffff" stroke="#90A5B1" stroke-width="1.2"/>
  <text x="662" y="130" text-anchor="middle" font-size="11.5" font-weight="600" fill="#0b2026">Model Training</text>
  <!-- Arrow from Model Training to Model Evaluation -->
  <path d="M662 150 L662 200" fill="none" stroke="#1B3139" stroke-width="1.8" marker-end="url(#mlc-arrow)"/>
  <!-- 6th step: Model evaluation -->
  <rect x="605" y="200" width="115" height="52" rx="8" fill="#ffffff" stroke="#90A5B1" stroke-width="1.2"/>
  <text x="662" y="230" text-anchor="middle" font-size="11.5" font-weight="600" fill="#0b2026">Model Evaluation</text>
  <!-- Top row arrows -->
  <path d="M158 124 L175 124" fill="none" stroke="#1B3139" stroke-width="1.8" marker-end="url(#mlc-arrow)"/>
  <path d="M290 124 L305 124" fill="none" stroke="#1B3139" stroke-width="1.8" marker-end="url(#mlc-arrow)"/>
  <path d="M420 124 L435 124" fill="none" stroke="#1B3139" stroke-width="1.8" marker-end="url(#mlc-arrow)"/>
  <path d="M585 124 L605 124" fill="none" stroke="#1B3139" stroke-width="1.8" marker-end="url(#mlc-arrow)"/>
<!-- Bottom subgroup: Deployment and Production — fill then dashed outline -->
  <rect x="30" y="310" width="700" height="200" rx="12" fill="#FF5F46" fill-opacity="0.06"/>
  <rect x="30" y="310" width="700" height="200" rx="12" fill="none" stroke="#FF5F46" stroke-width="1.5" stroke-dasharray="6 3"/>
  <text x="50" y="400"  font-size="13" font-weight="bold" fill="#0b2026">Deployment & Production</text>
  <text x="50" y="415"  font-size="11" fill="#618794">(continuously changing data)</text>
  <!-- 7th step: Model deployment -->
  <rect x="440" y="350" width="117" height="52" rx="8" fill="#ffffff" stroke="#90A5B1" stroke-width="1.5"/>
  <text x="500" y="380" text-anchor="middle" font-size="11.5" font-weight="600" fill="#0b2026">Model Deployment</text>
  <!-- 8th step: Model Monitoring -->
  <rect x="305" y="350" width="115" height="52" rx="8" fill="#ffffff" stroke="#90A5B1" stroke-width="1.5"/>
  <text x="360" y="380" text-anchor="middle" font-size="11.5" font-weight="600" fill="#0b2026">Model Monitoring</text>
  <!-- Arrow from Model Monitoring (top center) to Data Collection (bottom center) -->
  <path d="M362.5 350 L362.5 150" fill="none" stroke="#1B3139" stroke-width="1.8" marker-end="url(#mlc-arrow)"/>
<!-- Bottom row arrows -->
  <path d="M440 375 L420 375" fill="none" stroke="#1B3139" stroke-width="1.8" marker-end="url(#mlc-arrow)"/>
  <!-- Feedback: Model Evaluation → Data Preprocessing / Feature Engineering -->
  <path d="M 605 218 C 558 218 510 184 510 150" fill="none" stroke="#1B3139" stroke-width="1.8" stroke-dasharray="5 3" marker-end="url(#mlc-arrow)"/>
  <!-- Feedback: Model Evaluation → Data Collection -->
  <path d="M 605 234 C 555 274 362 274 362 150" fill="none" stroke="#1B3139" stroke-width="1.8" stroke-dasharray="5 3" marker-end="url(#mlc-arrow)"/>
  <!-- Forward: Model Evaluation → Model Deployment -->
  <path d="M 662 252 C 662 308 497 308 497 350" fill="none" stroke="#1B3139" stroke-width="1.8" marker-end="url(#mlc-arrow)"/>
</svg>
</div>

### A2. The Machine Learning Full Lifecycle

Building a model is only one stage in a larger workflow. The full lifecycle spans from defining the business problem through deployment and ongoing monitoring.

<div class="ft" style="max-width: 900px; margin: 18px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;"><style>.ft tbody tr { transition: background .15s ease; } .ft tbody tr:hover { background: #F4F8FB; } .ft code { background: #EEEDE9; border-radius: 4px; padding: 1px 5px; }</style><table style="border-collapse: separate; border-spacing: 0; width: 100%; background: #ffffff; border: 1px solid #E4E9F2; border-radius: 12px; overflow: hidden; box-shadow: 0 4px 16px rgba(27,49,57,0.08); font-size: 14.5px;"><thead><tr><th style="padding: 12px 16px; text-align: left; font-size: 12px; font-weight: 700; letter-spacing: 0.08em; text-transform: uppercase; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">Stage</th><th style="padding: 12px 16px; text-align: left; font-size: 12px; font-weight: 700; letter-spacing: 0.08em; text-transform: uppercase; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">Phase</th><th style="padding: 12px 16px; text-align: left; font-size: 12px; font-weight: 700; letter-spacing: 0.08em; text-transform: uppercase; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">Description</th></tr></thead><tbody><tr><td style="padding: 12px 16px; border-left: 5px solid #2272B4; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #EBF3FA; color: #2272B4; font-weight: 700; border: 1px solid #2272B440; white-space: nowrap;">Business Problem</span></td><td style="padding: 12px 16px; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><span style="display: inline-block; padding: 4px 12px; border-radius: 8px; background: #EBF3FA; color: #2272B4; font-weight: 600; border: 1px solid #2272B440;">Development</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">Define the business objective and frame it as a machine learning problem.</td></tr><tr><td style="padding: 12px 16px; border-left: 5px solid #2272B4; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #EBF3FA; color: #2272B4; font-weight: 700; border: 1px solid #2272B440; white-space: nowrap;">Define Success Criteria</span></td><td style="padding: 12px 16px; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><span style="display: inline-block; padding: 4px 12px; border-radius: 8px; background: #EBF3FA; color: #2272B4; font-weight: 600; border: 1px solid #2272B440;">Development</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">Establish measurable goals such as accuracy, latency, or revenue impact to determine whether the model meets business needs.</td></tr><tr><td style="padding: 12px 16px; border-left: 5px solid #00A972; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #E6F5EE; color: #00A972; font-weight: 700; border: 1px solid #00A97240; white-space: nowrap;">Data Collection &amp; Preprocessing</span></td><td style="padding: 12px 16px; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><span style="display: inline-block; padding: 4px 12px; border-radius: 8px; background: #EBF3FA; color: #2272B4; font-weight: 600; border: 1px solid #2272B440;">Development</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">Gather, clean, and transform relevant data into features suitable for model training.</td></tr><tr><td style="padding: 12px 16px; border-left: 5px solid #2272B4; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #EBF3FA; color: #2272B4; font-weight: 700; border: 1px solid #2272B440; white-space: nowrap;">Model Training</span></td><td style="padding: 12px 16px; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><span style="display: inline-block; padding: 4px 12px; border-radius: 8px; background: #EBF3FA; color: #2272B4; font-weight: 600; border: 1px solid #2272B440;">Development</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">Train candidate models on historical data, iterating on algorithms and hyperparameters to optimize performance.</td></tr><tr><td style="padding: 12px 16px; border-left: 5px solid #2272B4; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #EBF3FA; color: #2272B4; font-weight: 700; border: 1px solid #2272B440; white-space: nowrap;">Model Evaluation</span></td><td style="padding: 12px 16px; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><span style="display: inline-block; padding: 4px 12px; border-radius: 8px; background: #EBF3FA; color: #2272B4; font-weight: 600; border: 1px solid #2272B440;">Development</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">Assess model performance on held-out data and compare experiments to select the best candidate.</td></tr><tr><td style="padding: 12px 16px; border-left: 5px solid #FF5F46; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #FFF1EE; color: #FF5F46; font-weight: 700; border: 1px solid #FF5F4640; white-space: nowrap;">Model Deployment</span></td><td style="padding: 12px 16px; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><span style="display: inline-block; padding: 4px 12px; border-radius: 8px; background: #FFF1EE; color: #FF5F46; font-weight: 600; border: 1px solid #FF5F4640;">Production</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">Package and serve the selected model in a production environment to generate predictions on continuously arriving new data.</td></tr><tr><td style="padding: 12px 16px; border-left: 5px solid #FF5F46; vertical-align: top; "><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #FFF1EE; color: #FF5F46; font-weight: 700; border: 1px solid #FF5F4640; white-space: nowrap;">Model Monitoring</span></td><td style="padding: 12px 16px; vertical-align: top; "><span style="display: inline-block; padding: 4px 12px; border-radius: 8px; background: #FFF1EE; color: #FF5F46; font-weight: 600; border: 1px solid #FF5F4640;">Production</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; ">Track model performance on live data, detect drift or degradation, and feed insights back into retraining.</td></tr></tbody></table></div>

### A3. Data Preparation

Data preparation encompasses two distinct categories of work. Understanding the difference helps you sequence your pipeline correctly.

<div class="ft" style="max-width: 900px; margin: 18px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;"><style>.ft tbody tr { transition: background .15s ease; } .ft tbody tr:hover { background: #F4F8FB; } .ft code { background: #EEEDE9; border-radius: 4px; padding: 1px 5px; }</style><table style="border-collapse: separate; border-spacing: 0; width: 100%; background: #ffffff; border: 1px solid #E4E9F2; border-radius: 12px; overflow: hidden; box-shadow: 0 4px 16px rgba(27,49,57,0.08); font-size: 14.5px;"><thead><tr><th style="padding: 12px 16px; text-align: left; font-size: 12px; font-weight: 700; letter-spacing: 0.08em; text-transform: uppercase; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">Category</th><th style="padding: 12px 16px; text-align: left; font-size: 12px; font-weight: 700; letter-spacing: 0.08em; text-transform: uppercase; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">Goal</th><th style="padding: 12px 16px; text-align: left; font-size: 12px; font-weight: 700; letter-spacing: 0.08em; text-transform: uppercase; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">Typical Operations</th></tr></thead><tbody><tr><td style="padding: 12px 16px; border-left: 5px solid #2272B4; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #EBF3FA; color: #2272B4; font-weight: 700; border: 1px solid #2272B440; white-space: nowrap;">Cleaning &amp; formatting</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">Make the data <em>correct and usable</em></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">Fix data types, detect and treat invalid values and outliers, drop empty columns/rows, deduplicate</td></tr><tr><td style="padding: 12px 16px; border-left: 5px solid #00A972; vertical-align: top; "><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #E6F5EE; color: #00A972; font-weight: 700; border: 1px solid #00A97240; white-space: nowrap;">Feature engineering</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; ">Make the data <em>informative for the model</em></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; ">Imputation, numerical transforms (log, standardize), aggregations, encoding categoricals, generating new features</td></tr></tbody></table></div>

The same dataset usually requires both, and **the order matters**:

1. **Validate and clean** with rules that do not learn from the data (fix types, remove impossible values, deduplicate).
2. **Split** the data into training and test sets (section C).
3. **Fit** every step that learns from the data, such as imputers, encoders, scalers, and outlier thresholds, on the **training set only**.
4. **Apply** the fitted steps to the test set and to new data.

<div style="border-left: 4px solid #1B5162; background: #F8F9FC; padding: 16px 20px; border-radius: 4px; margin: 16px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;"><strong style="color: #1B5162;">Note</strong><div style="margin: 8px 0 0 0; color: #0b2026; line-height: 1.6;">Any step that calculates statistics, category lists, or thresholds from the data must be fit on the training set only. Otherwise information from the test set leaks into training, and your evaluation looks better than the model really is.</div></div>

### A4. Detecting and Treating Outliers

An **outlier** is a value that is far from the rest of the distribution. Some outliers are errors (a negative age, a typo that adds a zero); others are rare but real (a very large purchase). Treat the two differently: **fix or remove errors**, but think carefully before discarding genuine extreme values, because they may carry signal.

**Common detection methods**

<div class="ft" style="max-width: 900px; margin: 18px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;"><style>.ft tbody tr { transition: background .15s ease; } .ft tbody tr:hover { background: #F4F8FB; } .ft code { background: #EEEDE9; border-radius: 4px; padding: 1px 5px; }</style><table style="border-collapse: separate; border-spacing: 0; width: 100%; background: #ffffff; border: 1px solid #E4E9F2; border-radius: 12px; overflow: hidden; box-shadow: 0 4px 16px rgba(27,49,57,0.08); font-size: 14.5px;"><thead><tr><th style="padding: 12px 16px; text-align: left; font-size: 12px; font-weight: 700; letter-spacing: 0.08em; text-transform: uppercase; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">Method</th><th style="padding: 12px 16px; text-align: left; font-size: 12px; font-weight: 700; letter-spacing: 0.08em; text-transform: uppercase; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">Rule</th><th style="padding: 12px 16px; text-align: left; font-size: 12px; font-weight: 700; letter-spacing: 0.08em; text-transform: uppercase; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">When to use</th></tr></thead><tbody><tr><td style="padding: 12px 16px; border-left: 5px solid #2272B4; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #EBF3FA; color: #2272B4; font-weight: 700; border: 1px solid #2272B440; white-space: nowrap;">Domain rules</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">Values outside a valid range, such as <code>tenure &lt; 0</code> or <code>age &gt; 120</code></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">Always check these first; they catch data-entry errors</td></tr><tr><td style="padding: 12px 16px; border-left: 5px solid #4299E0; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #EEF6FD; color: #4299E0; font-weight: 700; border: 1px solid #4299E040; white-space: nowrap;">IQR (interquartile range)</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">Below Q1 &minus; 1.5&times;IQR or above Q3 + 1.5&times;IQR</td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">Skewed or non-normal distributions; robust to extreme values</td></tr><tr><td style="padding: 12px 16px; border-left: 5px solid #4299E0; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #EEF6FD; color: #4299E0; font-weight: 700; border: 1px solid #4299E040; white-space: nowrap;">Z-score</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">More than 3 standard deviations from the mean</td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">Roughly normal distributions</td></tr><tr><td style="padding: 12px 16px; border-left: 5px solid #7B61FF; vertical-align: top; "><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #F3F0FF; color: #7B61FF; font-weight: 700; border: 1px solid #7B61FF40; white-space: nowrap;">Visual inspection</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; ">Box plots and histograms (for example, generated with Genie Code)</td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; ">Exploratory analysis before choosing a rule</td></tr></tbody></table></div>

**Common treatments**

<div class="ft" style="max-width: 900px; margin: 18px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;"><style>.ft tbody tr { transition: background .15s ease; } .ft tbody tr:hover { background: #F4F8FB; } .ft code { background: #EEEDE9; border-radius: 4px; padding: 1px 5px; }</style><table style="border-collapse: separate; border-spacing: 0; width: 100%; background: #ffffff; border: 1px solid #E4E9F2; border-radius: 12px; overflow: hidden; box-shadow: 0 4px 16px rgba(27,49,57,0.08); font-size: 14.5px;"><thead><tr><th style="padding: 12px 16px; text-align: left; font-size: 12px; font-weight: 700; letter-spacing: 0.08em; text-transform: uppercase; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">Treatment</th><th style="padding: 12px 16px; text-align: left; font-size: 12px; font-weight: 700; letter-spacing: 0.08em; text-transform: uppercase; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">What it does</th><th style="padding: 12px 16px; text-align: left; font-size: 12px; font-weight: 700; letter-spacing: 0.08em; text-transform: uppercase; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">Trade-off</th></tr></thead><tbody><tr><td style="padding: 12px 16px; border-left: 5px solid #FF5F46; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #FFF1EE; color: #FF5F46; font-weight: 700; border: 1px solid #FF5F4640; white-space: nowrap;">Remove</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">Drop rows that contain invalid values</td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">Simple, but loses data. Use it for clear errors</td></tr><tr><td style="padding: 12px 16px; border-left: 5px solid #FFAB00; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #FFF7E6; color: #B26A00; font-weight: 700; border: 1px solid #FFAB0040; white-space: nowrap;">Cap (winsorize)</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">Clip values to a percentile range, such as the 1st to 99th</td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">Keeps every row while limiting the influence of extremes</td></tr><tr><td style="padding: 12px 16px; border-left: 5px solid #4299E0; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #EEF6FD; color: #4299E0; font-weight: 700; border: 1px solid #4299E040; white-space: nowrap;">Transform</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">Apply a log or square-root transform to compress a long tail</td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">Changes how the feature is interpreted</td></tr><tr><td style="padding: 12px 16px; border-left: 5px solid #2272B4; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #EBF3FA; color: #2272B4; font-weight: 700; border: 1px solid #2272B440; white-space: nowrap;">Use robust methods</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">Choose <code>RobustScaler</code> or median imputation instead of mean-based methods</td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">Reduces sensitivity without changing the data</td></tr><tr><td style="padding: 12px 16px; border-left: 5px solid #00A972; vertical-align: top; "><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #E6F5EE; color: #00A972; font-weight: 700; border: 1px solid #00A97240; white-space: nowrap;">Flag</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; ">Add an indicator column such as <code>is_outlier</code></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; ">Lets the model learn whether extreme values matter</td></tr></tbody></table></div>

In PySpark, `approxQuantile` computes the quartiles needed for an IQR rule at scale:

<div style="background: #F9F7F4; border: 1px solid #DCE0E2; border-radius: 8px; padding: 14px 18px; margin: 12px 0; overflow-x: auto; box-shadow: 0 2px 8px rgba(27,49,57,0.06);">
<pre style="margin: 0; font-family: ui-monospace, SFMono-Regular, Consolas, 'Liberation Mono', Menlo, monospace; font-size: 13px; color: #0b2026; white-space: pre-wrap;">
<code># Compute IQR bounds on the TRAINING set, then apply them to every split</code>
<code>q1, q3 = train_df.approxQuantile("MonthlyCharges", [0.25, 0.75], 0.01)</code>
<code>iqr = q3 - q1</code>
<code>lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr</code>
</br>
<code>from pyspark.sql import functions as F</code>
</br>
<code># Option 1: cap values to the bounds</code>
<code>capped_df = train_df.withColumn("MonthlyCharges",</code>
<code>    F.least(F.greatest(F.col("MonthlyCharges"), F.lit(lower)), F.lit(upper)))</code>
</br>
<code># Option 2: remove rows outside the bounds</code>
<code>filtered_df = train_df.filter(F.col("MonthlyCharges").between(lower, upper))</code></pre>
</div>

<div style="border-left: 4px solid #FFAB00; background: #FFF7E6; padding: 16px 20px; border-radius: 4px; margin: 16px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;"><strong style="color: #B26A00;">Caution</strong><div style="margin: 8px 0 0 0; color: #0b2026; line-height: 1.6;">Outlier thresholds are learned from the data, just like imputation values and scaler statistics. <b>Compute them on the training set only</b>, then apply the same bounds to the validation and test sets. The one exception is removing clear data errors, such as invalid values, which you can do before splitting. The demo in this module removes invalid values, and lecture 2.4 explains how outliers affect scaling.</div></div>

## B. Feature Engineering and Feature Selection

### B1. Feature Engineering

**Feature engineering** transforms raw data into a format that lets ML models extract patterns. The goal is not *more* features — it is *more informative* features.

Common operations include:

- **Numerical transforms** — log, standardize, bin
- **Encoding categoricals** — one-hot, label, target, embedding
- **Aggregations** — rolling means, group counts, recency
- **Derived features** — ratios, differences, interactions
- **Text / image / audio** — TF-IDF, embeddings, pretrained model outputs

#### Feature Extraction

A specialized case of feature engineering: derive a *smaller* set of features that better represent the underlying patterns. Examples include PCA, autoencoders, and learned embeddings.

### B2. Feature Selection

**Feature selection** chooses a smaller set of useful input features (columns). It is different from row-level cleaning, which decides which records to keep.

**Reasons to drop columns:**
- High percentage of missing values 
- Irrelevant or redundant features (for example, highly correlated with another column; whether to drop one depends on the model and on what the feature means)
- Not available at prediction time (using it would leak information)
- Computational efficiency

**Reasons to drop rows:**
- Too many missing fields per row
- Outliers that reflect data-quality issues, not signal

### B3. Leakage-Safe Spark ML Pipelines

On Databricks, you put every data-dependent step into a Spark ML **Pipeline**. Fitting the pipeline on the training data learns all the statistics at once (medians, scaling factors, category lists), and the fitted `PipelineModel` applies exactly the same logic to test data and to new data.

<div class="ft" style="max-width: 900px; margin: 18px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;"><style>.ft tbody tr { transition: background .15s ease; } .ft tbody tr:hover { background: #F4F8FB; } .ft code { background: #EEEDE9; border-radius: 4px; padding: 1px 5px; }</style><table style="border-collapse: separate; border-spacing: 0; width: 100%; background: #ffffff; border: 1px solid #E4E9F2; border-radius: 12px; overflow: hidden; box-shadow: 0 4px 16px rgba(27,49,57,0.08); font-size: 14.5px;"><thead><tr><th style="padding: 12px 16px; text-align: left; font-size: 12px; font-weight: 700; letter-spacing: 0.08em; text-transform: uppercase; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">Data</th><th style="padding: 12px 16px; text-align: left; font-size: 12px; font-weight: 700; letter-spacing: 0.08em; text-transform: uppercase; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">What happens</th></tr></thead><tbody><tr><td style="padding: 12px 16px; border-left: 5px solid #2272B4; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #EBF3FA; color: #2272B4; font-weight: 700; border: 1px solid #2272B440; white-space: nowrap;">Training data</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><code>pipeline.fit(train_df)</code> learns every statistic and returns a fitted <code>PipelineModel</code></td></tr><tr><td style="padding: 12px 16px; border-left: 5px solid #00A972; vertical-align: top; "><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #E6F5EE; color: #00A972; font-weight: 700; border: 1px solid #00A97240; white-space: nowrap;">Test and new data</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; "><code>pipeline_model.transform(df)</code> applies the fitted steps; nothing is re-learned</td></tr></tbody></table></div>

<div style="background: #F9F7F4; border: 1px solid #DCE0E2; border-radius: 8px; padding: 14px 18px; margin: 12px 0; overflow-x: auto; box-shadow: 0 2px 8px rgba(27,49,57,0.06);">
<pre style="margin: 0; font-family: ui-monospace, SFMono-Regular, Consolas, 'Liberation Mono', Menlo, monospace; font-size: 13px; color: #0b2026; white-space: pre-wrap;">
<code>from pyspark.ml import Pipeline</code>
<code>from pyspark.ml.feature import Imputer, VectorAssembler, StandardScaler, StringIndexer, OneHotEncoder</code>
</br>
<code>train_df, test_df = telco_df.randomSplit([0.8, 0.2], seed=42)   # 1. split first</code>
</br>
<code>pipeline = Pipeline(stages=[                                       # 2. define every data-dependent step</code>
<code>    Imputer(inputCols=[&quot;tenure&quot;, &quot;TotalCharges&quot;], outputCols=[&quot;tenure_imp&quot;, &quot;TotalCharges_imp&quot;], strategy=&quot;median&quot;),</code>
<code>    VectorAssembler(inputCols=[&quot;tenure_imp&quot;, &quot;TotalCharges_imp&quot;], outputCol=&quot;num_vec&quot;),</code>
<code>    StandardScaler(inputCol=&quot;num_vec&quot;, outputCol=&quot;num_scaled&quot;, withMean=True),</code>
<code>    StringIndexer(inputCol=&quot;Contract&quot;, outputCol=&quot;Contract_idx&quot;, handleInvalid=&quot;keep&quot;),</code>
<code>    OneHotEncoder(inputCol=&quot;Contract_idx&quot;, outputCol=&quot;Contract_ohe&quot;),</code>
<code>    VectorAssembler(inputCols=[&quot;num_scaled&quot;, &quot;Contract_ohe&quot;], outputCol=&quot;features&quot;),</code>
<code>])</code>
</br>
<code>pipeline_model = pipeline.fit(train_df)                            # 3. learn statistics from TRAIN only</code>
<code>train_prepared = pipeline_model.transform(train_df)                # 4. apply the same fitted pipeline</code>
<code>test_prepared = pipeline_model.transform(test_df)</code></pre>
</div>

<div style="border-left: 4px solid #00A972; background: #E6F5EE; padding: 16px 20px; border-radius: 4px; margin: 16px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;"><strong style="color: #00805A;">Databricks tip</strong><div style="margin: 8px 0 0 0; color: #0b2026; line-height: 1.6;">A fitted <code>PipelineModel</code> can be saved to a Unity Catalog volume and reloaded for inference, so production data is prepared exactly like the training data. The next lectures cover each stage (imputation, encoding, scaling), and the <i>2.6 Demo</i> builds a full pipeline.</div></div>

## C. Splitting Data

### C1. Why We Split

A model cannot be evaluated on the same data it was trained on — it would memorize rather than generalize. Splitting the dataset solves four interconnected problems:

<div class="ft" style="max-width: 900px; margin: 18px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;"><style>.ft tbody tr { transition: background .15s ease; } .ft tbody tr:hover { background: #F4F8FB; } .ft code { background: #EEEDE9; border-radius: 4px; padding: 1px 5px; }</style><table style="border-collapse: separate; border-spacing: 0; width: 100%; background: #ffffff; border: 1px solid #E4E9F2; border-radius: 12px; overflow: hidden; box-shadow: 0 4px 16px rgba(27,49,57,0.08); font-size: 14.5px;"><thead><tr><th style="padding: 12px 16px; text-align: left; font-size: 12px; font-weight: 700; letter-spacing: 0.08em; text-transform: uppercase; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">Need</th><th style="padding: 12px 16px; text-align: left; font-size: 12px; font-weight: 700; letter-spacing: 0.08em; text-transform: uppercase; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">What Splitting Gives You</th></tr></thead><tbody><tr><td style="padding: 12px 16px; border-left: 5px solid #00A972; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #E6F5EE; color: #00A972; font-weight: 700; border: 1px solid #00A97240; white-space: nowrap;">Avoid overfitting</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">A held-out test set the model never saw during training</td></tr><tr><td style="padding: 12px 16px; border-left: 5px solid #FFAB00; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #FFF7E6; color: #B26A00; font-weight: 700; border: 1px solid #FFAB0040; white-space: nowrap;">Hyperparameter tuning</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">A validation set you can iterate on without leaking the test set</td></tr><tr><td style="padding: 12px 16px; border-left: 5px solid #FF5F46; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #FFF1EE; color: #FF5F46; font-weight: 700; border: 1px solid #FF5F4640; white-space: nowrap;">Prevent data leakage</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">Transformations fit on the training set only, then applied to validation/test</td></tr><tr><td style="padding: 12px 16px; border-left: 5px solid #00A972; vertical-align: top; "><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #E6F5EE; color: #00A972; font-weight: 700; border: 1px solid #00A97240; white-space: nowrap;">Honest evaluation</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; ">A final test score that estimates real-world performance</td></tr></tbody></table></div>

### C2. Train / Validation / Test

The two most common split structures are shown below. The demos in this module use the simple 80/20 split.

<!-- SVG illustrating simple and validation splits -->
<svg width="100%" style="max-width: 600px; display: block;" viewBox="0 0 552 92" xmlns="http://www.w3.org/2000/svg" aria-label="Simple split: Train and Test">
  <!-- Simple split -->
  <text x="10" y="24" font-size="18" font-family="-apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif" font-weight="600" fill="#1B3139">Simple split:</text>
  <!-- 80% of 400px (train+test) = 320px, 20% = 80px, total = 400px, but scale to 552px width -->
  <rect x="20" y="40" width="352" height="44" rx="8" fill="#2272B4" stroke="#ffffff" stroke-width="2"/>
  <text x="150" y="65" font-size="16" font-family="-apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif" font-weight="600" fill="#fff">Train 80%</text>
  <rect x="374" y="40" width="88" height="44" rx="8" fill="#00A972" stroke="#ffffff" stroke-width="2"/>
  <text x="385" y="65" font-size="16" font-family="-apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif" font-weight="600" fill="#fff">Test 20%</text>
</svg>

Then, we validate and test the data as follows:
<svg width="100%" style="max-width: 600px; display: block;" viewBox="0 78 552 84" xmlns="http://www.w3.org/2000/svg" aria-label="Split with validation set: Train, Validation, and Test">
  <!-- Validation split -->
  <text x="10" y="100" font-size="18" font-family="-apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif" font-weight="600" fill="#1B3139">With validation set:</text>
  <rect x="20" y="112" width="264" height="44" rx="8" fill="#2272B4" stroke="#ffffff" stroke-width="2"/>
  <text x="130" y="138" font-size="16" font-family="-apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif" font-weight="600" fill="#fff">Train 60%</text>
  <rect x="284" y="112" width="88" height="44" rx="8" fill="#FFAB00" stroke="#ffffff" stroke-width="2"/>
  <text x="287" y="138" font-size="13" font-family="-apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif" font-weight="600" fill="#1B3139">Validate 20%</text>
  <rect x="374" y="112" width="88" height="44" rx="8" fill="#00A972" stroke="#ffffff" stroke-width="2"/>
  <text x="385" y="138" font-size="16" font-family="-apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif" font-weight="600" fill="#fff">Test 20%</text>
</svg>

In PySpark, use `randomSplit` to divide a DataFrame into proportional subsets. The `seed` parameter ensures reproducibility across runs.

### C3. Train / Test Example

#### Creating a Synthetic DataFrame

The following code creates a small synthetic dataset for demonstration purposes.  
`raw_df` is a Spark DataFrame with 1,000 rows, each containing:

</br>

<div style="background: #F9F7F4; border: 1px solid #DCE0E2; border-radius: 8px; padding: 14px 18px; margin: 12px 0; overflow-x: auto; box-shadow: 0 2px 8px rgba(27,49,57,0.06);">
<pre style="margin: 0; font-family: ui-monospace, SFMono-Regular, Consolas, 'Liberation Mono', Menlo, monospace; font-size: 13px; color: #0b2026; white-space: pre-wrap;">
<code>from pyspark.sql import functions as F</code>
</br>
<code>raw_df = (</code>
<code>    spark.range(1000)                                         # id: 0 to 999</code>
<code>    .withColumn(&quot;x&quot;, F.rand(seed=1))                          # a random feature</code>
<code>    .withColumn(&quot;label&quot;, (F.rand(seed=2) &gt; 0.5).cast(&quot;int&quot;))  # a random 0/1 label</code>
<code>)</code></pre>
</div>


- **id**: a unique integer identifier
- **x**: a random numeric feature
- **label**: a random binary class (0 or 1)

#### Splitting the Data

`randomSplit` divides `raw_df` into two subsets:

- **train_df**: about 80% of the data, used for model training
- **test_df**: about 20% of the data, held out for evaluation

The `seed` parameter makes the split reproducible. The proportions are approximate, so the counts will not be exactly 800 and 200.

<div style="background: #F9F7F4; border: 1px solid #DCE0E2; border-radius: 8px; padding: 14px 18px; margin: 12px 0; overflow-x: auto; box-shadow: 0 2px 8px rgba(27,49,57,0.06);">
<pre style="margin: 0; font-family: ui-monospace, SFMono-Regular, Consolas, 'Liberation Mono', Menlo, monospace; font-size: 13px; color: #0b2026; white-space: pre-wrap;">
<code>train_df, test_df = raw_df.randomSplit([0.8, 0.2], seed=42)
print(f"train: {train_df.count():,}   test: {test_df.count():,}")</code></pre>
</div>

### C4. Cross-Validation

When the dataset is small, a single train/validation split produces noisy estimates. **Cross-validation** rotates which fold serves as the validation set so that every row is evaluated exactly once.

<div class="ft" style="max-width: 900px; margin: 18px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;"><style>.ft tbody tr { transition: background .15s ease; } .ft tbody tr:hover { background: #F4F8FB; } .ft code { background: #EEEDE9; border-radius: 4px; padding: 1px 5px; }</style><table style="border-collapse: separate; border-spacing: 0; width: 100%; background: #ffffff; border: 1px solid #E4E9F2; border-radius: 12px; overflow: hidden; box-shadow: 0 4px 16px rgba(27,49,57,0.08); font-size: 14.5px;"><thead><tr><th style="padding: 12px 16px; text-align: left; font-size: 12px; font-weight: 700; letter-spacing: 0.08em; text-transform: uppercase; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">Strategy</th><th style="padding: 12px 16px; text-align: left; font-size: 12px; font-weight: 700; letter-spacing: 0.08em; text-transform: uppercase; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">When to Use</th></tr></thead><tbody><tr><td style="padding: 12px 16px; border-left: 5px solid #2272B4; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #EBF3FA; color: #2272B4; font-weight: 700; border: 1px solid #2272B440; white-space: nowrap;">K-Fold</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">Default for tabular data</td></tr><tr><td style="padding: 12px 16px; border-left: 5px solid #4299E0; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #EEF6FD; color: #4299E0; font-weight: 700; border: 1px solid #4299E040; white-space: nowrap;">Stratified K-Fold</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">Imbalanced classification — preserves class ratio per fold</td></tr><tr><td style="padding: 12px 16px; border-left: 5px solid #5A6F77; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #F0F2F4; color: #5A6F77; font-weight: 700; border: 1px solid #5A6F7740; white-space: nowrap;">Shuffle Split</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">When you want repeated random splits with a fixed test size</td></tr><tr><td style="padding: 12px 16px; border-left: 5px solid #7B61FF; vertical-align: top; "><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #F3F0FF; color: #7B61FF; font-weight: 700; border: 1px solid #7B61FF40; white-space: nowrap;">Nested CV</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; ">When you tune hyperparameters <em>and</em> report generalization — outer loop for evaluation, inner loop for tuning</td></tr></tbody></table></div>

In Spark ML, use `pyspark.ml.tuning.CrossValidator` (or `TrainValidationSplit` for a single split). Cross-validation is applied during model tuning, which is covered in the *Machine Learning Model Development* course.

**On Databricks:** Spark ML's `CrossValidator` performs **K-fold** cross-validation with random fold assignment. Stratified K-fold, shuffle split, and nested cross-validation are not built in: you can define your own folds with the `foldCol` parameter, or run nested loops yourself. The strategy names in the table are general ML terms that you will also see in single-machine libraries such as scikit-learn.

### C5. Time-Based Splits for Time-Ordered Data

`randomSplit` assigns each row to the training or test set at random. That works when rows are independent. When rows are ordered in time, such as transactions, sensor readings, or monthly customer snapshots, a random split puts rows from later dates into the training set and rows from earlier dates into the test set. The model then learns from information that did not exist yet when the test rows happened. This is called **look-ahead leakage**, and it makes the test score better than what you will see in production.

For time-ordered data, split **chronologically**: train on earlier data and evaluate on later data, the same way the model is used in production.

<div style="max-width: 900px; margin: 0 auto;">
<svg width="100%" viewBox="0 0 860 230" role="img" style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;">
  <title>Random split versus time-based split</title>
  <desc>Two horizontal timelines. The top one, labeled random split, mixes training and test rows across the whole time range, so the model can see the future. The bottom one, labeled time-based split, trains on all rows before a cutoff date and tests on rows after it, which mirrors how the model is used in production.</desc>
  <text x="34" y="42" text-anchor="start" font-size="14" font-weight="700" fill="#FF5F46">Random Split vs. Time-Based Split</text>
  <text x="40" y="78" font-size="12" font-weight="600" fill="#0b2026">Random split</text>
  <text x="40" y="94" font-size="11" fill="#98102A">leaks the future</text>
  <g>
    <rect x="200" y="64" width="60" height="32" rx="4" fill="#2272B4"/>
    <rect x="262" y="64" width="40" height="32" rx="4" fill="#00A972"/>
    <rect x="304" y="64" width="90" height="32" rx="4" fill="#2272B4"/>
    <rect x="396" y="64" width="40" height="32" rx="4" fill="#00A972"/>
    <rect x="438" y="64" width="120" height="32" rx="4" fill="#2272B4"/>
    <rect x="560" y="64" width="40" height="32" rx="4" fill="#00A972"/>
    <rect x="602" y="64" width="130" height="32" rx="4" fill="#2272B4"/>
    <rect x="734" y="64" width="80" height="32" rx="4" fill="#00A972"/>
  </g>
  <text x="40" y="146" font-size="12" font-weight="600" fill="#0b2026">Time-based split</text>
  <text x="40" y="162" font-size="11" fill="#00A972">mirrors production</text>
  <rect x="200" y="132" width="452" height="32" rx="4" fill="#2272B4"/>
  <text x="426" y="152" text-anchor="middle" font-size="12" font-weight="600" fill="#ffffff">Train: before cutoff</text>
  <rect x="656" y="132" width="158" height="32" rx="4" fill="#00A972"/>
  <text x="735" y="152" text-anchor="middle" font-size="12" font-weight="600" fill="#ffffff">Test: after cutoff</text>
  <path d="M654 124 L654 172" fill="none" stroke="#FF5F46" stroke-width="2" stroke-dasharray="4 3"/>
  <text x="654" y="188" text-anchor="middle" font-size="11" fill="#FF5F46">cutoff date</text>
  <path d="M200 200 L814 200" fill="none" stroke="#90A5B1" stroke-width="1.5"/>
  <text x="200" y="210" font-size="10" fill="#618794">older</text>
  <text x="814" y="210" text-anchor="end" font-size="10" fill="#618794">newer</text>
</svg>
</div>

Filter on the date column to split at a cutoff date:

<div style="background: #F9F7F4; border: 1px solid #DCE0E2; border-radius: 8px; padding: 14px 18px; margin: 12px 0; overflow-x: auto; box-shadow: 0 2px 8px rgba(27,49,57,0.06);">
<pre style="margin: 0; font-family: ui-monospace, SFMono-Regular, Consolas, 'Liberation Mono', Menlo, monospace; font-size: 13px; color: #0b2026; white-space: pre-wrap;">
<code>from pyspark.sql import functions as F</code>
</br>
<code># Train on everything before the cutoff date; test on everything on or after it</code>
<code>cutoff = &quot;2025-01-01&quot;</code>
<code>train_df = events_df.filter(F.col(&quot;event_date&quot;) &lt; cutoff)</code>
<code>test_df  = events_df.filter(F.col(&quot;event_date&quot;) &gt;= cutoff)</code></pre>
</div>

If you also need a validation set, use two cutoff dates so the three sets stay in time order: training first, then validation, then test.

<div style="background: #F9F7F4; border: 1px solid #DCE0E2; border-radius: 8px; padding: 14px 18px; margin: 12px 0; overflow-x: auto; box-shadow: 0 2px 8px rgba(27,49,57,0.06);">
<pre style="margin: 0; font-family: ui-monospace, SFMono-Regular, Consolas, 'Liberation Mono', Menlo, monospace; font-size: 13px; color: #0b2026; white-space: pre-wrap;">
<code>train_df = events_df.filter(F.col(&quot;event_date&quot;) &lt; &quot;2024-10-01&quot;)</code>
<code>val_df   = events_df.filter((F.col(&quot;event_date&quot;) &gt;= &quot;2024-10-01&quot;) &amp; (F.col(&quot;event_date&quot;) &lt; &quot;2025-01-01&quot;))</code>
<code>test_df  = events_df.filter(F.col(&quot;event_date&quot;) &gt;= &quot;2025-01-01&quot;)</code></pre>
</div>

**Choosing the cutoff:** pick it so that the test period looks like the period the model will predict in production (usually the most recent data), and so that it has enough rows of each class.

**Cross-validation for time-ordered data** uses the same idea. Instead of shuffling rows into folds, each fold trains on an expanding window of past data and validates on the period that follows (sometimes called *forward chaining*): train on months 1&ndash;3 and validate on month 4, then train on months 1&ndash;4 and validate on month 5, and so on. Each fold only learns from the past.

<div style="border-left: 4px solid #1B5162; background: #F8F9FC; padding: 16px 20px; border-radius: 4px; margin: 16px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;"><strong style="color: #1B5162;">On Databricks</strong><div style="margin: 8px 0 0 0; color: #0b2026; line-height: 1.6;">Spark ML's <code>CrossValidator</code> assigns rows to folds at random. Even with a custom fold column (<code>foldCol</code>), it trains each fold on all the other folds, including later ones, so it does not do forward chaining. For time-ordered data, build each training and validation window yourself with date filters, as shown above.</div></div>

<div style="border-left: 4px solid #FFAB00; background: #FFF7E6; padding: 16px 20px; border-radius: 4px; margin: 16px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;"><strong style="color: #B26A00;">Caution</strong><div style="margin: 8px 0 0 0; color: #0b2026; line-height: 1.6;"><b>Features can leak the future too.</b> A feature must only use information that was available at the time of each row. For example, a customer's total purchases computed over the full history leaks future purchases into older rows. Module 3 shows how point-in-time feature lookups prevent this.</div></div>

### C6. Group-Based Splits

When several rows belong to the same customer, patient, device, or account, a random split can put some of that customer's rows in the training set and others in the test set. The model then partly recognizes the customer instead of generalizing, and the test score is too optimistic. Keep each group entirely in one split: split the distinct IDs, then join them back to the rows.

<div style="background: #F9F7F4; border: 1px solid #DCE0E2; border-radius: 8px; padding: 14px 18px; margin: 12px 0; overflow-x: auto; box-shadow: 0 2px 8px rgba(27,49,57,0.06);">
<pre style="margin: 0; font-family: ui-monospace, SFMono-Regular, Consolas, 'Liberation Mono', Menlo, monospace; font-size: 13px; color: #0b2026; white-space: pre-wrap;">
<code># Split the distinct customers, not the rows, so a customer never appears in both sets</code>
<code>train_ids, test_ids = df.select(&quot;customer_id&quot;).distinct().randomSplit([0.8, 0.2], seed=42)</code>
<code>train_df = df.join(train_ids, &quot;customer_id&quot;)</code>
<code>test_df = df.join(test_ids, &quot;customer_id&quot;)</code></pre>
</div>

Because whole groups are assigned, the row split is only approximately 80/20, especially when some groups are much larger than others.

## D. Sampling Methods

### D1. Three Core Sampling Methods

Sampling creates a smaller, representative subset of a large dataset for exploration, prototyping, or reducing cost. **Sampling is not the same as splitting**: a split creates separate training and test sets, while a sample is a subset for analysis. The three core methods and their Spark APIs are summarized below.

<div class="ft" style="max-width: 900px; margin: 18px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;"><style>.ft tbody tr { transition: background .15s ease; } .ft tbody tr:hover { background: #F4F8FB; } .ft code { background: #EEEDE9; border-radius: 4px; padding: 1px 5px; }</style><table style="border-collapse: separate; border-spacing: 0; width: 100%; background: #ffffff; border: 1px solid #E4E9F2; border-radius: 12px; overflow: hidden; box-shadow: 0 4px 16px rgba(27,49,57,0.08); font-size: 14.5px;"><thead><tr><th style="padding: 12px 16px; text-align: left; font-size: 12px; font-weight: 700; letter-spacing: 0.08em; text-transform: uppercase; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">Method</th><th style="padding: 12px 16px; text-align: left; font-size: 12px; font-weight: 700; letter-spacing: 0.08em; text-transform: uppercase; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">When to Use</th><th style="padding: 12px 16px; text-align: left; font-size: 12px; font-weight: 700; letter-spacing: 0.08em; text-transform: uppercase; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">PySpark API</th></tr></thead><tbody><tr><td style="padding: 12px 16px; border-left: 5px solid #2272B4; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #EBF3FA; color: #2272B4; font-weight: 700; border: 1px solid #2272B440; white-space: nowrap;">Random</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">Default; every row is equally likely to be selected</td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><code>df.sample(withReplacement=False, fraction=0.05, seed=42)</code></td></tr><tr><td style="padding: 12px 16px; border-left: 5px solid #7B61FF; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #F3F0FF; color: #7B61FF; font-weight: 700; border: 1px solid #7B61FF40; white-space: nowrap;">Stratified</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">Imbalanced classes; preserve the class ratio in the sample</td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><code>df.sampleBy(col, fractions={...}, seed=42)</code></td></tr><tr><td style="padding: 12px 16px; border-left: 5px solid #FFAB00; vertical-align: top; "><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #FFF7E6; color: #B26A00; font-weight: 700; border: 1px solid #FFAB0040; white-space: nowrap;">Cluster</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; ">Natural groups (stores, regions); sample whole groups rather than individual rows</td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; ">Sample the distinct group IDs, then join them back to the data</td></tr></tbody></table></div>

The following cells show each method with Spark. Spark sampling fractions are **approximate**: each row is kept with the given probability.

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;"><strong style="color: #0d47a1;">Info</strong><div style="margin: 8px 0 0 0; color: #0b2026; line-height: 1.6;"><b>Sampling time-ordered data:</b> sample whole time windows (for example, specific weeks) rather than random rows, so the sample keeps the temporal patterns, such as trends and seasonality, that the model needs to learn. Also make sure features only use information that was available at prediction time. Module 3 returns to this idea with point-in-time feature lookups.</div></div>


### D2. Random Sampling

Random sampling is the most common and basic sampling technique in machine learning. It randomly selects data points from a dataset without following any specific pattern, assuming that each data point has an equal probability of being chosen.

<br>

<!-- SVG: Random sampling from rows -->
<svg width="400" height="200" viewBox="0 0 340 120" xmlns="http://www.w3.org/2000/svg">
  <!-- Four rows of people, each row has 4 different colors -->
  <g>
    <circle cx="30" cy="20" r="12" fill="#2272B4"/>
    <circle cx="70" cy="20" r="12" fill="#FF5F46"/>
    <circle cx="110" cy="20" r="12" fill="#00A972"/>
    <circle cx="150" cy="20" r="12" fill="#FFAB00"/>
  </g>
  <g>
    <circle cx="30" cy="50" r="12" fill="#FF5F46"/>
    <circle cx="70" cy="50" r="12" fill="#00A972"/>
    <circle cx="110" cy="50" r="12" fill="#FFAB00"/>
    <circle cx="150" cy="50" r="12" fill="#2272B4"/>
  </g>
  <g>
    <circle cx="30" cy="80" r="12" fill="#00A972"/>
    <circle cx="70" cy="80" r="12" fill="#FFAB00"/>
    <circle cx="110" cy="80" r="12" fill="#2272B4"/>
    <circle cx="150" cy="80" r="12" fill="#FF5F46"/>
  </g>
  <g>
    <circle cx="30" cy="110" r="12" fill="#FFAB00"/>
    <circle cx="70" cy="110" r="12" fill="#2272B4"/>
    <circle cx="110" cy="110" r="12" fill="#FF5F46"/>
    <circle cx="150" cy="110" r="12" fill="#00A972"/>
  </g>
  <!-- Horizontal arrow -->
  <defs>
    <marker id="arrowhead" markerWidth="8" markerHeight="8" refX="7" refY="4" orient="auto">
      <polygon points="0,0 8,4 0,8" fill="#1B3139"/>
    </marker>
  </defs>
  <line x1="190" y1="20" x2="280" y2="20" stroke="#1B3139" stroke-width="2" marker-end="url(#arrowhead)"/>
  <line x1="190" y1="50" x2="280" y2="50" stroke="#1B3139" stroke-width="2" marker-end="url(#arrowhead)"/>
  <line x1="190" y1="80" x2="280" y2="80" stroke="#1B3139" stroke-width="2" marker-end="url(#arrowhead)"/>
  <line x1="190" y1="110" x2="280" y2="110" stroke="#1B3139" stroke-width="2" marker-end="url(#arrowhead)"/>
  <!-- Selected person (on right of arrow), color matches one from each row -->
  <circle cx="300" cy="20" r="12" fill="#00A972" stroke="#1B5162" stroke-width="2"/>
  <circle cx="300" cy="50" r="12" fill="#FFAB00" stroke="#B26A00" stroke-width="2"/>
  <circle cx="300" cy="80" r="12" fill="#FF5F46" stroke="#98102A" stroke-width="2"/>
  <circle cx="300" cy="110" r="12" fill="#2272B4" stroke="#1B3139" stroke-width="2"/>
  <!-- Sampled label -->
  <text x="200" y="8" font-size="13" fill="#1B3139">Sampling</text>
</svg>

<br>

An example of random sampling of about 5% of the data with Spark:
<div style="background: #F9F7F4; border: 1px solid #DCE0E2; border-radius: 8px; padding: 14px 18px; margin: 12px 0; overflow-x: auto; box-shadow: 0 2px 8px rgba(27,49,57,0.06);">
<pre style="margin: 0; font-family: ui-monospace, SFMono-Regular, Consolas, 'Liberation Mono', Menlo, monospace; font-size: 13px; color: #0b2026; white-space: pre-wrap;">
<code># Random sample: keep about 5% of the rows</code>
<code>random_sample = df.sample(withReplacement=False, fraction=0.05, seed=42)</code>
<code>print(f&quot;Random sample: {random_sample.count():,} rows&quot;)</code></pre>
</div>


### D3. Stratified Sampling 


Stratified sampling is used when a dataset consists of subgroups, ensuring that samples are taken from each. The population is divided into homogeneous subgroups, known as strata, and an appropriate number of instances is selected from each stratum to ensure the sample accurately represents the entire population.

<br>

<!-- SVG: Random sampling from rows -->
<svg width="400" height="200" viewBox="0 0 340 120" xmlns="http://www.w3.org/2000/svg">
  <!-- Four rows of people, each row has 4 different colors -->
  <g>
    <circle cx="30" cy="20" r="12" fill="#2272B4"/>
    <circle cx="70" cy="20" r="12" fill="#2272B4"/>
    <circle cx="110" cy="20" r="12" fill="#FF5F46"/>
    <circle cx="150" cy="20" r="12" fill="#FF5F46"/>
  </g>
  <g>
    <circle cx="30" cy="50" r="12" fill="#2272B4"/>
    <circle cx="70" cy="50" r="12" fill="#2272B4"/>
    <circle cx="110" cy="50" r="12" fill="#FF5F46"/>
    <circle cx="150" cy="50" r="12" fill="#FF5F46"/>
  </g>
  <g>
    <circle cx="30" cy="80" r="12" fill="#00A972"/>
    <circle cx="70" cy="80" r="12" fill="#00A972"/>
    <circle cx="110" cy="80" r="12" fill="#FFAB00"/>
    <circle cx="150" cy="80" r="12" fill="#FFAB00"/>
  </g>
  <g>
    <circle cx="30" cy="110" r="12" fill="#00A972"/>
    <circle cx="70" cy="110" r="12" fill="#00A972"/>
    <circle cx="110" cy="110" r="12" fill="#FFAB00"/>
    <circle cx="150" cy="110" r="12" fill="#FFAB00"/>
  </g>
  <!-- Horizontal arrow -->
  <defs>
    <marker id="arrowhead" markerWidth="8" markerHeight="8" refX="7" refY="4" orient="auto">
      <polygon points="0,0 8,4 0,8" fill="#1B3139"/>
    </marker>
  </defs>
  <line x1="190" y1="20" x2="280" y2="20" stroke="#1B3139" stroke-width="2" marker-end="url(#arrowhead)"/>
  <line x1="190" y1="50" x2="280" y2="50" stroke="#1B3139" stroke-width="2" marker-end="url(#arrowhead)"/>
  <line x1="190" y1="80" x2="280" y2="80" stroke="#1B3139" stroke-width="2" marker-end="url(#arrowhead)"/>
  <line x1="190" y1="110" x2="280" y2="110" stroke="#1B3139" stroke-width="2" marker-end="url(#arrowhead)"/>
  <!-- Selected person (on right of arrow), color matches one from each row -->
  <circle cx="300" cy="20" r="12" fill="#2272B4" stroke="#1B3139" stroke-width="2"/>
  <circle cx="300" cy="50" r="12" fill="#FF5F46" stroke="#98102A" stroke-width="2"/>
  <circle cx="300" cy="80" r="12" fill="#00A972" stroke="#1B5162" stroke-width="2"/>
  <circle cx="300" cy="110" r="12" fill="#FFAB00" stroke="#B26A00" stroke-width="2"/>
  <!-- Sampled label -->
  <text x="200" y="8" font-size="13" fill="#1B3139">Sampling</text>
</svg>

An example of stratified sampling of the data. 

- `df` is a Spark DataFrame
- `label` is the column that defines the subgroups (strata)
- `sampleBy` takes a separate fraction for each subgroup, so you can also over- or under-sample a class


<div style="background: #F9F7F4; border: 1px solid #DCE0E2; border-radius: 8px; padding: 14px 18px; margin: 12px 0; overflow-x: auto; box-shadow: 0 2px 8px rgba(27,49,57,0.06);">
<pre style="margin: 0; font-family: ui-monospace, SFMono-Regular, Consolas, 'Liberation Mono', Menlo, monospace; font-size: 13px; color: #0b2026; white-space: pre-wrap;">
<code># Keep about 10% of each class; fractions are per class and approximate</code>
<code>stratified_sample = df.sampleBy(&quot;label&quot;, fractions={0: 0.1, 1: 0.1}, seed=42)</code>
<code>display(stratified_sample.groupBy(&quot;label&quot;).count())</code></pre>
</div>


### D4. Cluster Sampling 

Cluster sampling is a sampling technique where the population is divided into naturally occurring groups, or clusters, and a subset of these clusters is randomly selected for study. Instead of sampling individuals directly, researchers collect data from all individuals within the chosen clusters. 

<!-- SVG: Cluster sampling illustration -->
<svg width="600" height="260" viewBox="0 0 600 260" xmlns="http://www.w3.org/2000/svg">
  <!-- Large circle (population) -->
  <circle cx="120" cy="120" r="80" fill="#F9F7F4" stroke="#2272B4" stroke-width="3"/>
  <text x="85" y="30" font-size="16" fill="#2272B4" font-weight="bold">Population</text>
  <!-- 30 dots in large circle: 10 blue, 10 orange, 10 green -->
  <!-- Blue dots -->
  <circle cx="90" cy="70" r="5" fill="#2272B4"/>
  <circle cx="110" cy="65" r="5" fill="#2272B4"/>
  <circle cx="130" cy="75" r="5" fill="#2272B4"/>
  <circle cx="80" cy="100" r="5" fill="#2272B4"/>
  <circle cx="100" cy="110" r="5" fill="#2272B4"/>
  <circle cx="120" cy="100" r="5" fill="#2272B4"/>
  <circle cx="140" cy="90" r="5" fill="#2272B4"/>
  <circle cx="100" cy="140" r="5" fill="#2272B4"/>
  <circle cx="120" cy="130" r="5" fill="#2272B4"/>
  <circle cx="140" cy="120" r="5" fill="#2272B4"/>
  <!-- Orange dots -->
  <circle cx="160" cy="70" r="5" fill="#FF5F46"/>
  <circle cx="170" cy="100" r="5" fill="#FF5F46"/>
  <circle cx="150" cy="110" r="5" fill="#FF5F46"/>
  <circle cx="170" cy="130" r="5" fill="#FF5F46"/>
  <circle cx="160" cy="150" r="5" fill="#FF5F46"/>
  <circle cx="140" cy="150" r="5" fill="#FF5F46"/>
  <circle cx="120" cy="160" r="5" fill="#FF5F46"/>
  <circle cx="100" cy="170" r="5" fill="#FF5F46"/>
  <circle cx="80" cy="150" r="5" fill="#FF5F46"/>
  <circle cx="90" cy="170" r="5" fill="#FF5F46"/>
  <!-- Green dots -->
  <circle cx="70" cy="120" r="5" fill="#00A972"/>
  <circle cx="80" cy="130" r="5" fill="#00A972"/>
  <circle cx="110" cy="150" r="5" fill="#00A972"/>
  <circle cx="130" cy="170" r="5" fill="#00A972"/>
  <circle cx="150" cy="170" r="5" fill="#00A972"/>
  <circle cx="170" cy="150" r="5" fill="#00A972"/>
  <circle cx="170" cy="80" r="5" fill="#00A972"/>
  <circle cx="150" cy="80" r="5" fill="#00A972"/>
  <circle cx="130" cy="60" r="5" fill="#00A972"/>
  <circle cx="110" cy="80" r="5" fill="#00A972"/>
  <!-- Arrow to clusters -->
  <defs>
    <marker id="arrowhead" markerWidth="8" markerHeight="8" refX="7" refY="4" orient="auto">
      <polygon points="0,0 8,4 0,8" fill="#1B3139"/>
    </marker>
  </defs>
  <line x1="200" y1="120" x2="320" y2="60" stroke="#1B3139" stroke-width="2" marker-end="url(#arrowhead)"/>
  <line x1="200" y1="120" x2="320" y2="120" stroke="#1B3139" stroke-width="2" marker-end="url(#arrowhead)"/>
  <line x1="200" y1="120" x2="320" y2="180" stroke="#1B3139" stroke-width="2" marker-end="url(#arrowhead)"/>
  <!-- Cluster 1: blue dots -->
  <text x="400" y="65" font-size="13" fill="#2272B4"font-weight="bold" >Cluster 1</text>
  <circle cx="360" cy="60" r="30" fill="#F9F7F4" stroke="#2272B4" stroke-width="2"/>
  <circle cx="350" cy="50" r="5" fill="#2272B4"/>
  <circle cx="370" cy="50" r="5" fill="#2272B4"/>
  <circle cx="360" cy="40" r="5" fill="#2272B4"/>
  <circle cx="360" cy="60" r="5" fill="#2272B4"/>
  <circle cx="350" cy="70" r="5" fill="#2272B4"/>
  <circle cx="370" cy="70" r="5" fill="#2272B4"/>
  <circle cx="340" cy="60" r="5" fill="#2272B4"/>
  <circle cx="380" cy="60" r="5" fill="#2272B4"/>
  <circle cx="355" cy="80" r="5" fill="#2272B4"/>
  <circle cx="365" cy="80" r="5" fill="#2272B4"/>
  <!-- Cluster 2: orange dots -->
  <text x="400" y="125" font-size="13" fill="#FF5F46"font-weight="bold" >Cluster 2</text>
  <circle cx="360" cy="120" r="30" fill="#F9F7F4" stroke="#FF5F46" stroke-width="2"/>
  <circle cx="350" cy="110" r="5" fill="#FF5F46"/>
  <circle cx="370" cy="110" r="5" fill="#FF5F46"/>
  <circle cx="360" cy="100" r="5" fill="#FF5F46"/>
  <circle cx="360" cy="120" r="5" fill="#FF5F46"/>
  <circle cx="350" cy="130" r="5" fill="#FF5F46"/>
  <circle cx="370" cy="130" r="5" fill="#FF5F46"/>
  <circle cx="340" cy="120" r="5" fill="#FF5F46"/>
  <circle cx="380" cy="120" r="5" fill="#FF5F46"/>
  <circle cx="355" cy="140" r="5" fill="#FF5F46"/>
  <circle cx="365" cy="140" r="5" fill="#FF5F46"/>
  <!-- Cluster 3: green dots -->
  <text x="400" y="185" font-size="13" fill="#00A972"font-weight="bold" >Cluster 3</text>
  <circle cx="360" cy="180" r="30" fill="#F9F7F4" stroke="#00A972" stroke-width="2"/>
  <circle cx="350" cy="170" r="5" fill="#00A972"/>
  <circle cx="370" cy="170" r="5" fill="#00A972"/>
  <circle cx="360" cy="160" r="5" fill="#00A972"/>
  <circle cx="360" cy="180" r="5" fill="#00A972"/>
  <circle cx="350" cy="190" r="5" fill="#00A972"/>
  <circle cx="370" cy="190" r="5" fill="#00A972"/>
  <circle cx="340" cy="180" r="5" fill="#00A972"/>
  <circle cx="380" cy="180" r="5" fill="#00A972"/>
  <circle cx="355" cy="200" r="5" fill="#00A972"/>
  <circle cx="365" cy="200" r="5" fill="#00A972"/>
</svg>


</br>
Cluster sampling example: Sample whole groups, not individual rows.

- `df` is a Spark DataFrame
- `store_id` is the column that identifies each group
- Sample the distinct groups first, then join them back so that every row of a chosen group is kept

<div style="background: #F9F7F4; border: 1px solid #DCE0E2; border-radius: 8px; padding: 14px 18px; margin: 12px 0; overflow-x: auto; box-shadow: 0 2px 8px rgba(27,49,57,0.06);">
<pre style="margin: 0; font-family: ui-monospace, SFMono-Regular, Consolas, 'Liberation Mono', Menlo, monospace; font-size: 13px; color: #0b2026; white-space: pre-wrap;">
<code># Pick about 20% of the stores, then keep every row from those stores</code>
<code>chosen_stores = df.select(&quot;store_id&quot;).distinct().sample(fraction=0.2, seed=42)</code>
<code>cluster_sample = df.join(chosen_stores, on=&quot;store_id&quot;)</code></pre>
</div>

## E. Conclusion

In this lecture, you covered the conceptual foundations that underpin every demo and lab in this module:

1. **The Data Preparation Workflow** — clean with rules that do not learn from the data, split, then fit every data-dependent step on the training set only. Outliers should be detected with domain rules or statistical methods (IQR, z-score) and then removed, capped, transformed, or flagged, with thresholds computed on the training set.
2. **Feature Engineering and Selection** — feature engineering makes data informative, feature selection keeps the useful inputs, and a Spark ML Pipeline fit on the training data applies every learned step consistently to test and new data.
3. **Splitting Data** — transformations must be fit on the training set only, then applied to validation and test sets to prevent data leakage; cross-validation provides more reliable estimates when dataset size is limited. Time-ordered data must be split chronologically to avoid look-ahead leakage, and rows that belong to the same customer or device should stay in the same split.
4. **Sampling Methods** — Spark provides `sample` and `sampleBy` for random and stratified sampling at scale, and cluster samples are built by sampling group IDs. Sampling fractions are approximate, and a sample is not a replacement for a proper train/test split.

Together, these concepts form the foundation for robust, reproducible data preparation pipelines on Databricks — making it straightforward to clean, transform, split, and sample data before model training.


&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://www.databricks.com/legal/privacynotice" target="_blank">Privacy Notice</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Community</a>